Mount Google Drive & create folder structure

In [ ]:
import os
from datetime import datetime
from google.colab import drive

# --- Step 1: Mount Google Drive ---
print("📂 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted successfully!\n")

📂 Mounting Google Drive...
Mounted at /content/drive
✅ Google Drive mounted successfully!



In [ ]:
BASE_DIR = '/content/drive/MyDrive/Moshiko_Project'

FOLDERS = {
    'base'        : BASE_DIR,
    'models'      : f'{BASE_DIR}/models',           # Downloaded model weights
    'models_bf16' : f'{BASE_DIR}/models/bf16',      # Full precision weights
    'models_q8'   : f'{BASE_DIR}/models/q8',        # INT8 quantized weights
    'checkpoints' : f'{BASE_DIR}/checkpoints',      # Session resume points
    'outputs'     : f'{BASE_DIR}/outputs',           # All test results
    'audio_in'    : f'{BASE_DIR}/outputs/audio_input',   # Input audio files
    'audio_out'   : f'{BASE_DIR}/outputs/audio_output',  # Generated audio
    'benchmarks'  : f'{BASE_DIR}/outputs/benchmarks',    # Latency/speed results
    'comparisons' : f'{BASE_DIR}/outputs/comparisons',   # bf16 vs q8 results
    'logs'        : f'{BASE_DIR}/logs',             # Session logs
    'notebooks'   : f'{BASE_DIR}/notebooks',        # Notebook backups
}

# --- Step 2: Create all folders ---
print("📁 Creating project folder structure...")
for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)
    print(f"   {'✅ Exists' if os.path.exists(path) else '🆕 Created'}: {path.replace(BASE_DIR, 'Moshiko_Project')}")

# --- Step 3: Write a session log ---
session_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
log_file = f"{FOLDERS['logs']}/session_log.txt"
with open(log_file, 'a') as f:
    f.write(f"\n[{session_time}] New session started in Google Colab")

# --- Step 4: Update notebook timestamp in README ---
readme_path = f"{BASE_DIR}/README.txt"
readme_content = f"""
╔══════════════════════════════════════════════════════════╗
║           MOSHIKO RESEARCH PROJECT — README              ║
╠══════════════════════════════════════════════════════════╣
║  Last Session : {session_time}                           ║
╠══════════════════════════════════════════════════════════╣
║  FOLDER STRUCTURE                                        ║
║                                                          ║
║  Moshiko_Project/                                        ║
║  ├── models/                                             ║
║  │   ├── bf16/     ← Full precision model weights        ║
║  │   └── q8/       ← INT8 quantized model weights        ║
║  ├── checkpoints/  ← Session resume points               ║
║  ├── outputs/                                            ║
║  │   ├── audio_input/   ← Your test audio files          ║
║  │   ├── audio_output/  ← Generated speech outputs       ║
║  │   ├── benchmarks/    ← Latency & speed results        ║
║  │   └── comparisons/   ← bf16 vs q8 quality tests       ║
║  ├── logs/         ← Session activity logs               ║
║  └── notebooks/    ← Saved notebook backups              ║
╠══════════════════════════════════════════════════════════╣
║  HOW TO RESUME AFTER DISCONNECT:                         ║
║  1. Re-run Cell 1 (Drive mount)                          ║
║  2. Re-run Cell 2 (GPU check)                            ║
║  3. Re-run Cell 3 (Install packages)                     ║
║  4. Run Cell 4 (will auto-skip downloaded files)         ║
║  5. Re-run Cell 5 (Load model)                           ║
║  6. Run any test cell you need                           ║
╚══════════════════════════════════════════════════════════╝
"""
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"""
╔══════════════════════════════════════════════╗
║  ✅ CELL 1 COMPLETE                          ║
║  📁 Project root: MyDrive/Moshiko_Project/   ║
║  📝 README.txt created/updated               ║
║  📋 Session logged at {session_time[:10]}    ║
╚══════════════════════════════════════════════╝
""")

# Make FOLDERS accessible to all other cells
import builtins
builtins.FOLDERS = FOLDERS
builtins.BASE_DIR = BASE_DIR

📁 Creating project folder structure...
   ✅ Exists: Moshiko_Project
   ✅ Exists: Moshiko_Project/models
   ✅ Exists: Moshiko_Project/models/bf16
   ✅ Exists: Moshiko_Project/models/q8
   ✅ Exists: Moshiko_Project/checkpoints
   ✅ Exists: Moshiko_Project/outputs
   ✅ Exists: Moshiko_Project/outputs/audio_input
   ✅ Exists: Moshiko_Project/outputs/audio_output
   ✅ Exists: Moshiko_Project/outputs/benchmarks
   ✅ Exists: Moshiko_Project/outputs/comparisons
   ✅ Exists: Moshiko_Project/logs
   ✅ Exists: Moshiko_Project/notebooks

╔══════════════════════════════════════════════╗
║  ✅ CELL 1 COMPLETE                          ║
║  📁 Project root: MyDrive/Moshiko_Project/   ║
║  📝 README.txt created/updated               ║
║  📋 Session logged at 2026-04-03    ║
╚══════════════════════════════════════════════╝



CELL 2 — Check GPU & Environment

In [ ]:
import subprocess
import sys
import json
import os

print("🔍 Checking environment...\n")

# --- GPU Check ---
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        vram_free  = (torch.cuda.get_device_properties(0).total_memory
                      - torch.cuda.memory_allocated(0)) / 1e9
        print(f"  GPU      : {gpu_name}")
        print(f"  VRAM     : {vram_total:.1f} GB total | {vram_free:.1f} GB free")
    else:
        print("  ❌ NO GPU DETECTED!")
        print("  ⚠️  Go to: Runtime → Change runtime type → T4 GPU")
        raise SystemExit("Please enable GPU and re-run.")
except ImportError:
    gpu_name  = "Unknown (torch not installed yet)"
    vram_total = 0
    vram_free  = 0
    print("  ⚠️  PyTorch not installed yet — will check after Cell 3")

# --- Python Version ---
py_version = sys.version.split()[0]
print(f"  Python   : {py_version}")
py_ok = tuple(int(x) for x in py_version.split('.')[:2]) >= (3, 10)
print(f"  Python OK: {'✅ Yes' if py_ok else '❌ Needs 3.10+'}")

# --- Decide which model to use based on VRAM ---
if vram_total > 0:
    if vram_total >= 35:
        MODEL_PRECISION = 'bf16'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-bf16'
        precision_note  = 'Full precision (A100 detected)'
    elif vram_total >= 14:
        MODEL_PRECISION = 'q8'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
        precision_note  = 'INT8 quantized (T4 safe — uses ~14GB)'
    else:
        MODEL_PRECISION = 'q8'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
        precision_note  = '⚠️ Low VRAM — using INT8, may be tight'
else:
    MODEL_PRECISION = 'q8'
    MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
    precision_note  = 'Default (torch not yet installed)'

# Save config for later cells
builtins.MODEL_PRECISION = MODEL_PRECISION
builtins.MODEL_REPO      = MODEL_REPO

# --- Save environment info to Drive ---
try:
    env_info = {
        'session_time'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'gpu_name'         : gpu_name,
        'vram_total_gb'    : round(vram_total, 2),
        'vram_free_gb'     : round(vram_free,  2),
        'python_version'   : py_version,
        'selected_model'   : MODEL_REPO,
        'model_precision'  : MODEL_PRECISION,
    }
    env_path = f"{FOLDERS['logs']}/environment_info.json"
    with open(env_path, 'w') as f:
        json.dump(env_info, f, indent=2)
    print(f"\n  💾 Environment info saved to Drive")
except Exception as e:
    print(f"  ⚠️  Could not save env info: {e} (run Cell 1 first)")

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 2 COMPLETE                                 ║
║  🎯 Selected model : {MODEL_REPO:<32}               ║
║  📊 Precision      : {MODEL_PRECISION:<32}          ║
║  📝 Note           : {precision_note:<32}           ║
╚══════════════════════════════════════════════════════╝
""")


🔍 Checking environment...

  GPU      : Tesla T4
  VRAM     : 15.6 GB total | 15.6 GB free
  Python   : 3.12.13
  Python OK: ✅ Yes

  💾 Environment info saved to Drive

╔══════════════════════════════════════════════════════╗
║  ✅ CELL 2 COMPLETE                                 ║
║  🎯 Selected model : kyutai/moshiko-pytorch-q8                      ║
║  📊 Precision      : q8                                        ║
║  📝 Note           : INT8 quantized (T4 safe — uses ~14GB)           ║
╚══════════════════════════════════════════════════════╝



Install All Dependencies

In [ ]:
import subprocess, sys, importlib

def install(package, import_name=None):
    """Install a package and verify it imported correctly."""
    name = import_name or package.split('==')[0].replace('-','_')
    try:
        importlib.import_module(name)
        print(f"  ✅ Already installed : {package}")
    except ImportError:
        print(f"  📥 Installing        : {package} ...", end='', flush=True)
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', package],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(" Done ✅")
        else:
            print(f" FAILED ❌\n    Error: {result.stderr[:200]}")

print("📦 Installing required packages...\n")

packages = [
    ('moshi',          'moshi'),
    ('huggingface_hub','huggingface_hub'),
    ('torchaudio',     'torchaudio'),
    ('soundfile',      'soundfile'),
    ('numpy',          'numpy'),
    ('scipy',          'scipy'),
    ('matplotlib',     'matplotlib'),
    ('IPython',        'IPython'),
    ('tqdm',           'tqdm'),
]

for pkg, imp in packages:
    install(pkg, imp)

# --- Verify critical imports ---
print("\n🔍 Verifying critical imports...")
critical = ['torch', 'moshi', 'huggingface_hub', 'torchaudio', 'soundfile']
all_ok = True
for mod in critical:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'unknown')
        print(f"  ✅ {mod:<20} v{ver}")
    except ImportError:
        print(f"  ❌ {mod:<20} IMPORT FAILED — re-run this cell")
        all_ok = False

if all_ok:
    print("""
╔══════════════════════════════════════════╗
║  ✅  All packages OK                     ║
╚══════════════════════════════════════════╝
""")
    print("➡️  Ready for Cell 4")
else:
    print("""
╔══════════════════════════════════════════╗
║  ❌ Some packages failed — re-run cell  ║
╚══════════════════════════════════════════╝
""")

📦 Installing required packages...

  📥 Installing        : moshi ... Done ✅
  ✅ Already installed : huggingface_hub
  ✅ Already installed : torchaudio
  ✅ Already installed : soundfile
  ✅ Already installed : numpy
  ✅ Already installed : scipy
  ✅ Already installed : matplotlib
  ✅ Already installed : IPython
  ✅ Already installed : tqdm

🔍 Verifying critical imports...
  ✅ torch                v2.10.0+cu128
  ✅ moshi                v0.2.13
  ✅ huggingface_hub      v0.36.2
  ✅ torchaudio           v2.10.0+cu128
  ✅ soundfile            v0.13.1

╔══════════════════════════════════════════╗
║  ✅  All packages OK                     ║
╚══════════════════════════════════════════╝

➡️  Ready for Cell 4


Download Models

In [ ]:
import os, json, time
from datetime import datetime
from huggingface_hub import hf_hub_download
from tqdm import tqdm

# ---- Checkpoint helpers ----
CHECKPOINT_FILE = f"{FOLDERS['checkpoints']}/download_checkpoint.json"

def load_checkpoint():
    """Load which files have already been downloaded."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {'downloaded': {}, 'last_updated': None}

def save_checkpoint(ckpt):
    """Save progress after each file download."""
    ckpt['last_updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(ckpt, f, indent=2)

def download_file(repo_id, filename, local_dir, ckpt, label):
    """
    Download a single file from HuggingFace Hub.
    Skips if already downloaded (checked via checkpoint).
    """
    key = f"{repo_id}/{filename}"
    local_path = os.path.join(local_dir, filename)

    # Check checkpoint
    if key in ckpt['downloaded'] and os.path.exists(local_path):
        size_mb = os.path.getsize(local_path) / 1e6
        print(f"  ⏭️  SKIP (already downloaded): {label} ({size_mb:.0f} MB)")
        return local_path

    print(f"  📥 Downloading: {label} ...", flush=True)
    start = time.time()
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=local_dir,
            local_dir_use_symlinks=False,
        )
        elapsed  = time.time() - start
        size_mb  = os.path.getsize(path) / 1e6
        print(f"     ✅ Done — {size_mb:.0f} MB in {elapsed:.0f}s")

        # Save to checkpoint
        ckpt['downloaded'][key] = {
            'path': path,
            'size_mb': round(size_mb, 1),
            'downloaded_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        save_checkpoint(ckpt)
        return path
    except Exception as e:
        print(f"     ❌ FAILED: {e}")
        print(f"     💡 Re-run this cell to retry.")
        return None

# ---- Load existing checkpoint ----
ckpt = load_checkpoint()
if ckpt['last_updated']:
    print(f"📋 Checkpoint found — last updated: {ckpt['last_updated']}")
    print(f"   Already downloaded: {len(ckpt['downloaded'])} file(s)\n")
else:
    print("📋 No checkpoint found — starting fresh download\n")

# ---- Files to download ----
# We download BOTH q8 and bf16 Mimi weights + Moshiko weights
# q8 = for running on T4 | bf16 = for quality comparison test
files_to_download = [
    # Q8 model (primary — fits T4)
    ('kyutai/moshiko-pytorch-q8', 'mimi_weight.pt',  FOLDERS['models_q8'],  'Mimi Codec (q8)'),
    ('kyutai/moshiko-pytorch-q8', 'moshiko-q8.safetensors', FOLDERS['models_q8'], 'Moshiko LM (q8)'),
    # BF16 Mimi only (for comparison — Mimi is small ~200MB)
    ('kyutai/moshiko-pytorch-bf16', 'mimi_weight.pt', FOLDERS['models_bf16'], 'Mimi Codec (bf16)'),
]

print("⬇️  Starting downloads...\n")
downloaded_paths = {}
for repo, fname, local_dir, label in files_to_download:
    path = download_file(repo, fname, local_dir, ckpt, label)
    if path:
        downloaded_paths[label] = path

# ---- Summary ----
total_files = len(ckpt['downloaded'])
print(f"""
╔══════════════════════════════════════════════════════════╗
║  ✅ CELL 4 COMPLETE                                      ║
║  📦 Total files in Drive : {total_files} file(s)                    ║
║  💾 Checkpoint saved     : {CHECKPOINT_FILE.split('/')[-1]:<26}  ║
║                                                          ║
║  If this cell was interrupted:                           ║
║  → Re-run Cell 4 — it will resume where it stopped      ║
╚══════════════════════════════════════════════════════════╝
""")


📋 No checkpoint found — starting fresh download

⬇️  Starting downloads...

  📥 Downloading: Mimi Codec (q8) ...
     ❌ FAILED: 404 Client Error. (Request ID: Root=1-69cfdd8e-6e5978c6717aa26822c392b7;4012610f-3370-40b0-bb9e-230b9bff5a61)

Entry Not Found for url: https://huggingface.co/kyutai/moshiko-pytorch-q8/resolve/main/mimi_weight.pt.
     💡 Re-run this cell to retry.
  📥 Downloading: Moshiko LM (q8) ...
     ❌ FAILED: 404 Client Error. (Request ID: Root=1-69cfdd8e-37a45c84456dc448285e075d;56bc014f-034d-4477-93b9-f47432c492a1)

Entry Not Found for url: https://huggingface.co/kyutai/moshiko-pytorch-q8/resolve/main/moshiko-q8.safetensors.
     💡 Re-run this cell to retry.
  📥 Downloading: Mimi Codec (bf16) ...
     ❌ FAILED: 404 Client Error. (Request ID: Root=1-69cfdd8f-3028b88216ac524a0aefdf08;3cfb0e41-5628-4775-9a93-b181e593411a)

Entry Not Found for url: https://huggingface.co/kyutai/moshiko-pytorch-bf16/resolve/main/mimi_weight.pt.
     💡 Re-run this cell to retry.

╔══════════

In [ ]:

import os
BASE_DIR = "/content/drive/MyDrive/Moshiko_Project"  # Adjust path as needed
FOLDERS = {
    'models_q8': os.path.join(BASE_DIR, 'models_q8'),
    'models_bf16': os.path.join(BASE_DIR, 'models_bf16'),
    'checkpoints': os.path.join(BASE_DIR, 'checkpoints'),
}
# Create directories
for path in FOLDERS.values():
    os.makedirs(path, exist_ok=True)
print("✅ Folders ready:", list(FOLDERS.values()))

✅ Folders ready: ['/content/drive/MyDrive/Moshiko_Project/models_q8', '/content/drive/MyDrive/Moshiko_Project/models_bf16', '/content/drive/MyDrive/Moshiko_Project/checkpoints']


In [ ]:
# ============================================================
# CELL 4 — Download Models with Checkpoint System (FIXED)
# ============================================================
# PURPOSE: Download Moshiko model weights to Google Drive.
#          Uses a checkpoint file to track what's already
#          downloaded — so disconnects don't restart downloads.
#
# ⏱️  First run: ~10-20 min (large model files)
# ✅ Resume-safe: Already downloaded files are SKIPPED
# ============================================================

import os, json, time
from datetime import datetime
from huggingface_hub import hf_hub_download, login
from tqdm import tqdm

# ---- Optional: Authenticate if repos are gated ----
# login()  # Uncomment if you get 401 errors or repos are private

# ---- Checkpoint helpers ----
CHECKPOINT_FILE = f"{FOLDERS['checkpoints']}/download_checkpoint.json"

def load_checkpoint():
    """Load which files have already been downloaded."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {'downloaded': {}, 'last_updated': None}

def save_checkpoint(ckpt):
    """Save progress after each file download."""
    ckpt['last_updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    os.makedirs(os.path.dirname(CHECKPOINT_FILE), exist_ok=True)
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(ckpt, f, indent=2)

def download_file(repo_id, filename, local_dir, ckpt, label):
    """
    Download a single file from HuggingFace Hub.
    Skips if already downloaded (checked via checkpoint).
    """
    key = f"{repo_id}/{filename}"
    local_path = os.path.join(local_dir, filename)

    # Check checkpoint
    if key in ckpt['downloaded'] and os.path.exists(local_path):
        size_mb = os.path.getsize(local_path) / 1e6
        print(f"  ⏭️  SKIP (already downloaded): {label} ({size_mb:.0f} MB)")
        return local_path

    print(f"  📥 Downloading: {label} ...", flush=True)
    start = time.time()
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=local_dir,
            local_dir_use_symlinks=False,
        )
        elapsed  = time.time() - start
        size_mb  = os.path.getsize(path) / 1e6
        print(f"     ✅ Done — {size_mb:.0f} MB in {elapsed:.0f}s")

        # Save to checkpoint
        ckpt['downloaded'][key] = {
            'path': path,
            'size_mb': round(size_mb, 1),
            'downloaded_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        save_checkpoint(ckpt)
        return path
    except Exception as e:
        print(f"     ❌ FAILED: {e}")
        print(f"     💡 Re-run this cell to retry.")
        return None

# ---- Load existing checkpoint ----
ckpt = load_checkpoint()
if ckpt['last_updated']:
    print(f"📋 Checkpoint found — last updated: {ckpt['last_updated']}")
    print(f"   Already downloaded: {len(ckpt['downloaded'])} file(s)\n")
else:
    print("📋 No checkpoint found — starting fresh download\n")

# ---- Files to download ----
# ✅ CORRECTED FILENAMES based on actual HuggingFace repo contents
# Repo: https://huggingface.co/kyutai/moshiko-pytorch-q8
# Repo: https://huggingface.co/kyutai/moshiko-pytorch-bf16

files_to_download = [
    # ── Q8 models (primary — optimized for T4 GPU, ~8.4 GB total) ──
    ('kyutai/moshiko-pytorch-q8',
     'model.q8.safetensors',
     FOLDERS['models_q8'],
     'Mimi Codec (q8)'),

    ('kyutai/moshiko-pytorch-q8',
     'tokenizer-e351c8d8-checkpoint125.safetensors',
     FOLDERS['models_q8'],
     'Moshiko LM (q8)'),

    # ── BF16 Mimi only (for comparison — ~385 MB) ──
    ('kyutai/moshiko-pytorch-bf16',
     'tokenizer-e351c8d8-checkpoint125.safetensors',
     FOLDERS['models_bf16'],
     'Mimi Codec (bf16)'),

    # ── Optional: BF16 Moshiko LM (uncomment if needed — ~15.4 GB) ──
    # ('kyutai/moshiko-pytorch-bf16',
    #  'model.safetensors',
    #  FOLDERS['models_bf16'],
    #  'Moshiko LM (bf16)'),
]

print("⬇️  Starting downloads...\n")
downloaded_paths = {}
for repo, fname, local_dir, label in files_to_download:
    # Ensure target directory exists
    os.makedirs(local_dir, exist_ok=True)
    path = download_file(repo, fname, local_dir, ckpt, label)
    if path:
        downloaded_paths[label] = path

# ---- Summary ----
total_files = len(ckpt['downloaded'])
total_size_mb = sum(info.get('size_mb', 0) for info in ckpt['downloaded'].values())

print(f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║  ✅ CELL 4 COMPLETE                                                           ║
║  📦 Total files downloaded : {total_files} file(s){" " * 20}                  ║
║  💾 Total size on Drive    : {total_size_mb:>.0f} MB{" " * 22}                ║
║  💾 Checkpoint saved       : {CHECKPOINT_FILE.split('/')[-1]:<26}             ║
║                                                                               ║
║  📁 Download locations:                                                       ║
""")
for label, path in downloaded_paths.items():
    size = os.path.getsize(path) / 1e6
    print(f"     • {label:<25} → {size:>.0f} MB")
print(f"""
║                                                          ║
║  If this cell was interrupted:                           ║
║  → Re-run Cell 4 — it will resume where it stopped       ║
╚══════════════════════════════════════════════════════════╝
""")
print("➡️  Ready for Cell 5")

📋 No checkpoint found — starting fresh download

⬇️  Starting downloads...

  📥 Downloading: Mimi Codec (q8) ...


model.q8.safetensors:   0%|          | 0.00/8.01G [00:00<?, ?B/s]

     ✅ Done — 8009 MB in 78s
  📥 Downloading: Moshiko LM (q8) ...


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

     ✅ Done — 385 MB in 5s
  📥 Downloading: Mimi Codec (bf16) ...


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

     ✅ Done — 385 MB in 5s

╔════════════════════════════════════════════════════════════════════════════════╗
║  ✅ CELL 4 COMPLETE                                                           ║
║  📦 Total files downloaded : 3 file(s)                                      ║
║  💾 Total size on Drive    : 8778 MB                                      ║
║  💾 Checkpoint saved       : download_checkpoint.json               ║
║                                                                               ║
║  📁 Download locations:                                                       ║

     • Mimi Codec (q8)           → 8009 MB
     • Moshiko LM (q8)           → 385 MB
     • Mimi Codec (bf16)         → 385 MB

║                                                          ║
║  If this cell was interrupted:                           ║
║  → Re-run Cell 4 — it will resume where it stopped       ║
╚══════════════════════════════════════════════════════════╝

➡️  Ready for Cell 5


Load Moshiko Model Into Memory

In [ ]:
import moshi
print(f"📦 moshi version: {moshi.__version__}")
print("🔍 Available in moshi.models.loaders:", [x for x in dir(moshi.models.loaders) if not x.startswith('_')])

📦 moshi version: 0.2.13
🔍 Available in moshi.models.loaders: ['BaseConditioner', 'CheckpointInfo', 'ConditionFuser', 'ConditionProvider', 'DEFAULT_REPO', 'EntryNotFoundError', 'FRAME_RATE', 'LMModel', 'MIMI_NAME', 'MOSHI_NAME', 'MOSHI_Q8_NAME', 'MimiModel', 'Path', 'SAMPLE_RATE', 'SEANetDecoder', 'SEANetEncoder', 'SplitResidualVectorQuantizer', 'TEXT_TOKENIZER_NAME', 'dataclass', 'field', 'get_condition_fuser', 'get_conditioner', 'get_conditioner_provider', 'get_lora_moshi', 'get_mimi', 'get_moshi_lm', 'hf_get', 'hf_hub_download', 'json', 'load_file', 'replace_all_linear_with_lora', 'replace_lora_with_linear', 'sentencepiece', 'torch', 'tp', 'transformer', 'warnings']


In [ ]:
import torch, time, os, builtins, importlib
from safetensors.torch import load_file

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")
if DEVICE == 'cpu':
    print("  ⚠️  WARNING: CPU mode will be very slow! Enable GPU in Runtime settings.")

# ---- Locate model files ----
Q8_DIR = FOLDERS['models_q8']
mimi_path  = os.path.join(Q8_DIR, 'tokenizer-e351c8d8-checkpoint125.safetensors')
moshi_path = os.path.join(Q8_DIR, 'model.q8.safetensors')

for label, path in [('Mimi Codec', mimi_path), ('Moshiko LM (q8)', moshi_path)]:
    if not os.path.exists(path):
        print(f"  ❌ Missing: {label} at {path}")
        raise FileNotFoundError(f"Missing model file: {path}")
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Load Mimi (Audio Codec) ----
print("\n⏳ Loading Mimi audio codec...")
start = time.time()
from moshi.models import loaders
mimi = loaders.get_mimi(mimi_path, device=DEVICE)  # get_mimi is stable across versions
mimi.set_num_codebooks(8)
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- Load Moshiko LM (Auto-Detect & Load) ----
print("\n⏳ Loading Moshiko language model (q8)...")
start = time.time()
moshi_lm = None

# 1️⃣ Try official loaders (covers v0.1+)
if hasattr(loaders, 'load_moshi'):
    moshi_lm = loaders.load_moshi(moshi_path, device=DEVICE)
elif hasattr(loaders, 'get_moshi'):
    moshi_lm = loaders.get_moshi(moshi_path, device=DEVICE)
elif hasattr(loaders, 'load_model'):
    moshi_lm = loaders.load_model(moshi_path, device=DEVICE)

# 2️⃣ Fallback: Direct class instantiation
if moshi_lm is None:
    print("  ⚠️  Standard loader not found. Searching for model class...")
    import moshi.models as models_mod
    # Find classes that look like the main LM
    candidates = [name for name in dir(models_mod) if any(k in name for k in ['Moshi', 'LM', 'Model']) and name[0].isupper()]
    if candidates:
        ModelCls = getattr(models_mod, candidates[0])
        moshi_lm = ModelCls()
        state_dict = load_file(moshi_path)
        moshi_lm.load_state_dict(state_dict, strict=False)
        moshi_lm = moshi_lm.to(DEVICE)
        print(f"  ✅ Loaded via direct class: {candidates[0]}")
    else:
        raise ImportError(
            "❌ Cannot find Moshi LM loader/class.\n"
            "💡 Fix: Run this in a new cell:\n"
            "   !pip uninstall moshi -y\n"
            "   !pip install git+https://github.com/kyutai-labs/moshi.git\n"
            "Then restart runtime & re-run Cell 5."
        )

moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- VRAM usage ----
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    print(f"\n  📊 VRAM used    : {torch.cuda.memory_allocated()/1e9:.1f} GB")
    print(f"  📊 VRAM reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB")

# Global scope
builtins.mimi     = mimi
builtins.moshi_lm = moshi_lm
builtins.DEVICE   = DEVICE

print(f"""
╔{'═'*76}╗
║  ✅ CELL 5 COMPLETE — Models loaded!                              ║
║  🎵 Mimi codec   : Ready (audio tokenizer)                        ║
║  🧠 Moshiko LM   : Ready (speech-text model)                      ║
║  🖥️  Device       : {DEVICE:<48} ║
╚{'═'*76}╝
""")


🖥️  Using device: cuda
  ✅ Found: Mimi Codec (385 MB)
  ✅ Found: Moshiko LM (q8) (8009 MB)

⏳ Loading Mimi audio codec...
  ✅ Mimi loaded in 0.7s

⏳ Loading Moshiko language model (q8)...
  ⚠️  Standard loader not found. Searching for model class...


TypeError: Can't instantiate abstract class CompressionModel without an implementation for abstract methods '_init_streaming_state', 'cardinality', 'channels', 'decode', 'decode_latent', 'encode', 'forward', 'frame_rate', 'frame_size', 'num_codebooks', 'sample_rate', 'set_num_codebooks', 'total_codebooks'

In [ ]:
# 🛠️ INSTALL OFFICIAL MOSHI PACKAGE (Kyutai)
!pip uninstall moshi -y -q
!pip install git+https://github.com/kyutai-labs/moshi.git -q
print("✅ Installed official moshi package. Please RESTART RUNTIME now.")

ERROR: git+https://github.com/kyutai-labs/moshi.git does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
✅ Installed official moshi package. Please RESTART RUNTIME now.


In [ ]:
import torch, time, os, builtins
from moshi.models import loaders

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")

# ---- Locate model files ----
Q8_DIR = FOLDERS['models_q8']
mimi_path  = os.path.join(Q8_DIR, 'tokenizer-e351c8d8-checkpoint125.safetensors')
moshi_path = os.path.join(Q8_DIR, 'model.q8.safetensors')

# Verify files
for label, path in [('Mimi Codec', mimi_path), ('Moshiko LM (q8)', moshi_path)]:
    assert os.path.exists(path), f"❌ Missing: {path}"
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Load Mimi ----
print("\n⏳ Loading Mimi audio codec...")
start = time.time()
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.set_num_codebooks(8)
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- Load Moshiko LM ✅ USING get_moshi_lm (exists in your v0.2.13) ----
print("\n⏳ Loading Moshiko language model (q8)...")
start = time.time()
moshi_lm = loaders.get_moshi_lm(moshi_path, device=DEVICE)
moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- VRAM check ----
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    print(f"\n  📊 VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Global scope
builtins.mimi = mimi
builtins.moshi_lm = moshi_lm
builtins.DEVICE = DEVICE

print(f"""
╔{'═'*76}╗
║  ✅ CELL 5 COMPLETE — Models loaded!                              ║
║  🎵 Mimi codec   : Ready                                          ║
║  🧠 Moshiko LM   : Ready                                          ║
║  🖥️  Device       : {DEVICE:<48} ║
╚{'═'*76}╝
""")


🖥️  Using device: cuda
  ✅ Found: Mimi Codec (385 MB)
  ✅ Found: Moshiko LM (q8) (8009 MB)

⏳ Loading Mimi audio codec...
  ✅ Mimi loaded in 0.7s

⏳ Loading Moshiko language model (q8)...


ImportError: cannot import name 'size_hint' from 'torch.fx.experimental.symbolic_shapes' (/usr/local/lib/python3.12/dist-packages/torch/fx/experimental/symbolic_shapes.py)

 **Day-4/4/26**

In [1]:
!pwd

/content


In [2]:
import os
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [3]:
!pwd

/content


In [4]:
!ls

drive  sample_data


In [9]:
BASE_DIR = '/content/drive/MyDrive/Moshiko_Project'

FOLDERS = {
    'base'        : BASE_DIR,
    'models'      : f'{BASE_DIR}/models',           # Downloaded model weights
    'models_bf16' : f'{BASE_DIR}/models/bf16',      # Full precision weights
    'models_q8'   : f'{BASE_DIR}/models/q8',        # INT8 quantized weights
    'checkpoints' : f'{BASE_DIR}/checkpoints',      # Session resume points
    'outputs'     : f'{BASE_DIR}/outputs',           # All test results
    'audio_in'    : f'{BASE_DIR}/outputs/audio_input',   # Input audio files
    'audio_out'   : f'{BASE_DIR}/outputs/audio_output',  # Generated audio
    'benchmarks'  : f'{BASE_DIR}/outputs/benchmarks',    # Latency/speed results
    'comparisons' : f'{BASE_DIR}/outputs/comparisons',   # bf16 vs q8 results
    'logs'        : f'{BASE_DIR}/logs',             # Session logs
    'notebooks'   : f'{BASE_DIR}/notebooks',        # Notebook backups
}

# --- Step 2: Create all folders ---
print("📁 Creating project folder structure...")
for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)
    print(f"   {'✅ Exists' if os.path.exists(path) else '🆕 Created'}: {path.replace(BASE_DIR, 'Moshiko_Project')}")

# --- Step 3: Write a session log ---
session_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
log_file = f"{FOLDERS['logs']}/session_log.txt"
with open(log_file, 'a') as f:
    f.write(f"\n[{session_time}] New session started in Google Colab")

# --- Step 4: Update notebook timestamp in README ---
readme_path = f"{BASE_DIR}/README.txt"
readme_content = f"""
╔══════════════════════════════════════════════════════════╗
║           MOSHIKO RESEARCH PROJECT — README              ║
╠══════════════════════════════════════════════════════════╣
║  Last Session : {session_time}                           ║
╠══════════════════════════════════════════════════════════╣
║  FOLDER STRUCTURE                                        ║
║                                                          ║
║  Moshiko_Project/                                        ║
║  ├── models/                                             ║
║  │   ├── bf16/     ← Full precision model weights        ║
║  │   └── q8/       ← INT8 quantized model weights        ║
║  ├── checkpoints/  ← Session resume points               ║
║  ├── outputs/                                            ║
║  │   ├── audio_input/   ← Your test audio files          ║
║  │   ├── audio_output/  ← Generated speech outputs       ║
║  │   ├── benchmarks/    ← Latency & speed results        ║
║  │   └── comparisons/   ← bf16 vs q8 quality tests       ║
║  ├── logs/         ← Session activity logs               ║
║  └── notebooks/    ← Saved notebook backups              ║
╠══════════════════════════════════════════════════════════╣
║  HOW TO RESUME AFTER DISCONNECT:                         ║
║  1. Re-run Cell 1 (Drive mount)                          ║
║  2. Re-run Cell 2 (GPU check)                            ║
║  3. Re-run Cell 3 (Install packages)                     ║
║  4. Run Cell 4 (will auto-skip downloaded files)         ║
║  5. Re-run Cell 5 (Load model)                           ║
║  6. Run any test cell you need                           ║
╚══════════════════════════════════════════════════════════╝
"""
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"""
╔══════════════════════════════════════════════╗
║  ✅ CELL 1 COMPLETE                          ║
║  📁 Project root: MyDrive/Moshiko_Project/   ║
║  📝 README.txt created/updated               ║
║  📋 Session logged at {session_time[:10]}    ║
╚══════════════════════════════════════════════╝
""")

# Make FOLDERS accessible to all other cells
import builtins
builtins.FOLDERS = FOLDERS
builtins.BASE_DIR = BASE_DIR

📁 Creating project folder structure...
   ✅ Exists: Moshiko_Project
   ✅ Exists: Moshiko_Project/models
   ✅ Exists: Moshiko_Project/models/bf16
   ✅ Exists: Moshiko_Project/models/q8
   ✅ Exists: Moshiko_Project/checkpoints
   ✅ Exists: Moshiko_Project/outputs
   ✅ Exists: Moshiko_Project/outputs/audio_input
   ✅ Exists: Moshiko_Project/outputs/audio_output
   ✅ Exists: Moshiko_Project/outputs/benchmarks
   ✅ Exists: Moshiko_Project/outputs/comparisons
   ✅ Exists: Moshiko_Project/logs
   ✅ Exists: Moshiko_Project/notebooks

╔══════════════════════════════════════════════╗
║  ✅ CELL 1 COMPLETE                          ║
║  📁 Project root: MyDrive/Moshiko_Project/   ║
║  📝 README.txt created/updated               ║
║  📋 Session logged at 2026-04-04    ║
╚══════════════════════════════════════════════╝



In [10]:
import subprocess
import sys
import json
import os

print("🔍 Checking environment...\n")

# --- GPU Check ---
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        vram_free  = (torch.cuda.get_device_properties(0).total_memory
                      - torch.cuda.memory_allocated(0)) / 1e9
        print(f"  GPU      : {gpu_name}")
        print(f"  VRAM     : {vram_total:.1f} GB total | {vram_free:.1f} GB free")
    else:
        print("  ❌ NO GPU DETECTED!")
        print("  ⚠️  Go to: Runtime → Change runtime type → T4 GPU")
        raise SystemExit("Please enable GPU and re-run.")
except ImportError:
    gpu_name  = "Unknown (torch not installed yet)"
    vram_total = 0
    vram_free  = 0
    print("  ⚠️  PyTorch not installed yet — will check after Cell 3")

# --- Python Version ---
py_version = sys.version.split()[0]
print(f"  Python   : {py_version}")
py_ok = tuple(int(x) for x in py_version.split('.')[:2]) >= (3, 10)
print(f"  Python OK: {'✅ Yes' if py_ok else '❌ Needs 3.10+'}")

# --- Decide which model to use based on VRAM ---
if vram_total > 0:
    if vram_total >= 35:
        MODEL_PRECISION = 'bf16'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-bf16'
        precision_note  = 'Full precision (A100 detected)'
    elif vram_total >= 14:
        MODEL_PRECISION = 'q8'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
        precision_note  = 'INT8 quantized (T4 safe — uses ~14GB)'
    else:
        MODEL_PRECISION = 'q8'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
        precision_note  = '⚠️ Low VRAM — using INT8, may be tight'
else:
    MODEL_PRECISION = 'q8'
    MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
    precision_note  = 'Default (torch not yet installed)'

# Save config for later cells
builtins.MODEL_PRECISION = MODEL_PRECISION
builtins.MODEL_REPO      = MODEL_REPO

# --- Save environment info to Drive ---
try:
    env_info = {
        'session_time'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'gpu_name'         : gpu_name,
        'vram_total_gb'    : round(vram_total, 2),
        'vram_free_gb'     : round(vram_free,  2),
        'python_version'   : py_version,
        'selected_model'   : MODEL_REPO,
        'model_precision'  : MODEL_PRECISION,
    }
    env_path = f"{FOLDERS['logs']}/environment_info.json"
    with open(env_path, 'w') as f:
        json.dump(env_info, f, indent=2)
    print(f"\n  💾 Environment info saved to Drive")
except Exception as e:
    print(f"  ⚠️  Could not save env info: {e} (run Cell 1 first)")

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 2 COMPLETE                                 ║
║  🎯 Selected model : {MODEL_REPO:<32}               ║
║  📊 Precision      : {MODEL_PRECISION:<32}          ║
║  📝 Note           : {precision_note:<32}           ║
╚══════════════════════════════════════════════════════╝
""")


🔍 Checking environment...

  GPU      : Tesla T4
  VRAM     : 15.6 GB total | 15.6 GB free
  Python   : 3.12.13
  Python OK: ✅ Yes

  💾 Environment info saved to Drive

╔══════════════════════════════════════════════════════╗
║  ✅ CELL 2 COMPLETE                                 ║
║  🎯 Selected model : kyutai/moshiko-pytorch-q8                      ║
║  📊 Precision      : q8                                        ║
║  📝 Note           : INT8 quantized (T4 safe — uses ~14GB)           ║
╚══════════════════════════════════════════════════════╝



In [11]:
import subprocess, sys, importlib

def install(package, import_name=None):
    """Install a package and verify it imported correctly."""
    name = import_name or package.split('==')[0].replace('-','_')
    try:
        importlib.import_module(name)
        print(f"  ✅ Already installed : {package}")
    except ImportError:
        print(f"  📥 Installing        : {package} ...", end='', flush=True)
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', package],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(" Done ✅")
        else:
            print(f" FAILED ❌\n    Error: {result.stderr[:200]}")

print("📦 Installing required packages...\n")

packages = [
    ('moshi',          'moshi'),
    ('huggingface_hub','huggingface_hub'),
    ('torchaudio',     'torchaudio'),
    ('soundfile',      'soundfile'),
    ('numpy',          'numpy'),
    ('scipy',          'scipy'),
    ('matplotlib',     'matplotlib'),
    ('IPython',        'IPython'),
    ('tqdm',           'tqdm'),
]

for pkg, imp in packages:
    install(pkg, imp)

# --- Verify critical imports ---
print("\n🔍 Verifying critical imports...")
critical = ['torch', 'moshi', 'huggingface_hub', 'torchaudio', 'soundfile']
all_ok = True
for mod in critical:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'unknown')
        print(f"  ✅ {mod:<20} v{ver}")
    except ImportError:
        print(f"  ❌ {mod:<20} IMPORT FAILED — re-run this cell")
        all_ok = False

if all_ok:
    print("""
╔══════════════════════════════════════════╗
║  ✅  All packages OK                     ║
╚══════════════════════════════════════════╝
""")
    print("➡️  Ready for Cell 4")
else:
    print("""
╔══════════════════════════════════════════╗
║  ❌ Some packages failed — re-run cell  ║
╚══════════════════════════════════════════╝
""")

📦 Installing required packages...

  📥 Installing        : moshi ... Done ✅
  ✅ Already installed : huggingface_hub
  ✅ Already installed : torchaudio
  ✅ Already installed : soundfile
  ✅ Already installed : numpy
  ✅ Already installed : scipy
  ✅ Already installed : matplotlib
  ✅ Already installed : IPython
  ✅ Already installed : tqdm

🔍 Verifying critical imports...
  ✅ torch                v2.10.0+cu128
  ✅ moshi                v0.2.13
  ✅ huggingface_hub      v0.36.2
  ✅ torchaudio           v2.10.0+cu128
  ✅ soundfile            v0.13.1

╔══════════════════════════════════════════╗
║  ✅  All packages OK                     ║
╚══════════════════════════════════════════╝

➡️  Ready for Cell 4


In [12]:
import os, json, time
from datetime import datetime
from huggingface_hub import hf_hub_download, login
from tqdm import tqdm

# ---- Optional: Authenticate if repos are gated ----
# login()  # Uncomment if you get 401 errors or repos are private

# ---- Checkpoint helpers ----
CHECKPOINT_FILE = f"{FOLDERS['checkpoints']}/download_checkpoint.json"

def load_checkpoint():
    """Load which files have already been downloaded."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {'downloaded': {}, 'last_updated': None}

def save_checkpoint(ckpt):
    """Save progress after each file download."""
    ckpt['last_updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    os.makedirs(os.path.dirname(CHECKPOINT_FILE), exist_ok=True)
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(ckpt, f, indent=2)

def download_file(repo_id, filename, local_dir, ckpt, label):
    """
    Download a single file from HuggingFace Hub.
    Skips if already downloaded (checked via checkpoint).
    """
    key = f"{repo_id}/{filename}"
    local_path = os.path.join(local_dir, filename)

    # Check checkpoint
    if key in ckpt['downloaded'] and os.path.exists(local_path):
        size_mb = os.path.getsize(local_path) / 1e6
        print(f"  ⏭️  SKIP (already downloaded): {label} ({size_mb:.0f} MB)")
        return local_path

    print(f"  📥 Downloading: {label} ...", flush=True)
    start = time.time()
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=local_dir,
            local_dir_use_symlinks=False,
        )
        elapsed  = time.time() - start
        size_mb  = os.path.getsize(path) / 1e6
        print(f"     ✅ Done — {size_mb:.0f} MB in {elapsed:.0f}s")

        # Save to checkpoint
        ckpt['downloaded'][key] = {
            'path': path,
            'size_mb': round(size_mb, 1),
            'downloaded_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        save_checkpoint(ckpt)
        return path
    except Exception as e:
        print(f"     ❌ FAILED: {e}")
        print(f"     💡 Re-run this cell to retry.")
        return None

# ---- Load existing checkpoint ----
ckpt = load_checkpoint()
if ckpt['last_updated']:
    print(f"📋 Checkpoint found — last updated: {ckpt['last_updated']}")
    print(f"   Already downloaded: {len(ckpt['downloaded'])} file(s)\n")
else:
    print("📋 No checkpoint found — starting fresh download\n")

# ---- Files to download ----
# ✅ CORRECTED FILENAMES based on actual HuggingFace repo contents
# Repo: https://huggingface.co/kyutai/moshiko-pytorch-q8
# Repo: https://huggingface.co/kyutai/moshiko-pytorch-bf16

files_to_download = [
    # ── Q8 models (primary — optimized for T4 GPU, ~8.4 GB total) ──
    ('kyutai/moshiko-pytorch-q8',
     'model.q8.safetensors',
     FOLDERS['models_q8'],
     'Mimi Codec (q8)'),

    ('kyutai/moshiko-pytorch-q8',
     'tokenizer-e351c8d8-checkpoint125.safetensors',
     FOLDERS['models_q8'],
     'Moshiko LM (q8)'),

    # ── BF16 Mimi only (for comparison — ~385 MB) ──
    ('kyutai/moshiko-pytorch-bf16',
     'tokenizer-e351c8d8-checkpoint125.safetensors',
     FOLDERS['models_bf16'],
     'Mimi Codec (bf16)'),

    # ── Optional: BF16 Moshiko LM (uncomment if needed — ~15.4 GB) ──
    # ('kyutai/moshiko-pytorch-bf16',
    #  'model.safetensors',
    #  FOLDERS['models_bf16'],
    #  'Moshiko LM (bf16)'),
]

print("⬇️  Starting downloads...\n")
downloaded_paths = {}
for repo, fname, local_dir, label in files_to_download:
    # Ensure target directory exists
    os.makedirs(local_dir, exist_ok=True)
    path = download_file(repo, fname, local_dir, ckpt, label)
    if path:
        downloaded_paths[label] = path

# ---- Summary ----
total_files = len(ckpt['downloaded'])
total_size_mb = sum(info.get('size_mb', 0) for info in ckpt['downloaded'].values())

print(f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║  ✅ CELL 4 COMPLETE                                                           ║
║  📦 Total files downloaded : {total_files} file(s){" " * 20}                  ║
║  💾 Total size on Drive    : {total_size_mb:>.0f} MB{" " * 22}                ║
║  💾 Checkpoint saved       : {CHECKPOINT_FILE.split('/')[-1]:<26}             ║
║                                                                               ║
║  📁 Download locations:                                                       ║
""")
for label, path in downloaded_paths.items():
    size = os.path.getsize(path) / 1e6
    print(f"     • {label:<25} → {size:>.0f} MB")
print(f"""
║                                                          ║
║  If this cell was interrupted:                           ║
║  → Re-run Cell 4 — it will resume where it stopped       ║
╚══════════════════════════════════════════════════════════╝
""")
print("➡️  Ready for Cell 5")

📋 Checkpoint found — last updated: 2026-04-03 15:48:57
   Already downloaded: 3 file(s)

⬇️  Starting downloads...

  📥 Downloading: Mimi Codec (q8) ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


model.q8.safetensors:   0%|          | 0.00/8.01G [00:00<?, ?B/s]

     ✅ Done — 8009 MB in 56s
  📥 Downloading: Moshiko LM (q8) ...


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

     ✅ Done — 385 MB in 4s
  📥 Downloading: Mimi Codec (bf16) ...


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

     ✅ Done — 385 MB in 5s

╔════════════════════════════════════════════════════════════════════════════════╗
║  ✅ CELL 4 COMPLETE                                                           ║
║  📦 Total files downloaded : 3 file(s)                                      ║
║  💾 Total size on Drive    : 8778 MB                                      ║
║  💾 Checkpoint saved       : download_checkpoint.json               ║
║                                                                               ║
║  📁 Download locations:                                                       ║

     • Mimi Codec (q8)           → 8009 MB
     • Moshiko LM (q8)           → 385 MB
     • Mimi Codec (bf16)         → 385 MB

║                                                          ║
║  If this cell was interrupted:                           ║
║  → Re-run Cell 4 — it will resume where it stopped       ║
╚══════════════════════════════════════════════════════════╝

➡️  Ready for Cell 5


In [13]:
import torch
print(torch.cuda.is_available())        # Should print: True
print(torch.cuda.get_device_name(0))    # Should print: Tesla T4
print(torch.version.cuda)               # Should print: 12.x or 11.x

True
Tesla T4
12.8


In [14]:
import torch, time, os, builtins
from moshi.models import loaders, LMGen

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")
if DEVICE == 'cpu':
    print("  ⚠️  WARNING: CPU mode will be very slow!")

# ---- Locate model files ----
Q8_DIR   = FOLDERS['models_q8']
BF16_DIR = FOLDERS['models_bf16']

mimi_path  = os.path.join(Q8_DIR, 'tokenizer-e351c8d8-checkpoint125.safetensors')
moshi_path = os.path.join(Q8_DIR, 'model.q8.safetensors')

# Check files exist
for label, path in [('Mimi Codec', mimi_path), ('Moshiko LM (q8)', moshi_path)]:
    if not os.path.exists(path):
        print(f"  ❌ Missing: {label} at {path}")
        print("  → Re-run Cell 4 to download missing files.")
        raise FileNotFoundError(f"Missing: {path}")
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Load Mimi (Audio Codec) ----
print("\n⏳ Loading Mimi audio codec...")
start = time.time()
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.set_num_codebooks(8)
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- Load Moshiko LM ----
# CORRECT function: get_moshi_lm()  (NOT get_moshi or load_moshi)
print("\n⏳ Loading Moshiko language model (q8)...")
start = time.time()
moshi_lm = loaders.get_moshi_lm(moshi_path, device=DEVICE)
moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- Wrap in LMGen (required for streaming inference) ----
# LMGen handles sampling parameters for generation
print("\n⏳ Wrapping in LMGen (streaming generator)...")
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
print("  ✅ LMGen ready")

# ---- VRAM usage ----
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"\n  📊 VRAM used    : {allocated:.1f} GB")
    print(f"  📊 VRAM reserved: {reserved:.1f} GB")

# ---- Make globally accessible to all other cells ----
builtins.mimi     = mimi
builtins.moshi_lm = moshi_lm
builtins.lm_gen   = lm_gen
builtins.DEVICE   = DEVICE

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 5 COMPLETE — Models loaded!                 ║
║  🎵 Mimi codec   : Ready (audio tokenizer)           ║
║  🧠 Moshiko LM   : Ready (speech-text model)         ║
║  🎛️  LMGen        : Ready (streaming inference)       ║
║  🖥️  Device       : {DEVICE:<34}║
╚══════════════════════════════════════════════════════╝
""")
print("➡️  Ready for TEST Cells (6, 7, 8, 9)")

🖥️  Using device: cuda
  ✅ Found: Mimi Codec (385 MB)
  ✅ Found: Moshiko LM (q8) (8009 MB)

⏳ Loading Mimi audio codec...
  ✅ Mimi loaded in 2.5s

⏳ Loading Moshiko language model (q8)...


ImportError: cannot import name 'size_hint' from 'torch.fx.experimental.symbolic_shapes' (/usr/local/lib/python3.12/dist-packages/torch/fx/experimental/symbolic_shapes.py)

In [15]:
# ============================================================
# CELL 3b — Fix PyTorch / Moshi Version Conflict
# ============================================================
# Run this ONCE, then Runtime → Restart Runtime
# After restart: run Cell 1 → 2 → 3 → 3b-check → 4 → 5
# ============================================================

import subprocess, sys

print("🔍 Checking current versions...")
import torch
print(f"  Current PyTorch : {torch.__version__}")
print(f"  Current CUDA    : {torch.version.cuda}")

print("\n📥 Installing compatible versions...")

# Step 1: Uninstall current moshi
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'moshi', '-y'],
               capture_output=True)
print("  ✅ Uninstalled old moshi")

# Step 2: Install PyTorch 2.4 (last version confirmed working with moshi)
print("  📥 Installing PyTorch 2.4.0 + CUDA 12.1 (takes ~2 min)...")
result = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.4.0', 'torchaudio==2.4.0',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], capture_output=True, text=True)
if result.returncode == 0:
    print("  ✅ PyTorch 2.4.0 installed")
else:
    print(f"  ❌ Failed: {result.stderr[-300:]}")

# Step 3: Install moshi from GitHub (latest, compatible with torch 2.4)
print("  📥 Installing moshi from GitHub...")
result = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/kyutai-labs/moshi.git#egg=moshi&subdirectory=moshi'
], capture_output=True, text=True)
if result.returncode == 0:
    print("  ✅ moshi (GitHub latest) installed")
else:
    print(f"  ❌ Failed: {result.stderr[-300:]}")

# Step 4: Reinstall safetensors
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'safetensors'],
               capture_output=True)
print("  ✅ safetensors installed")

print("""
╔══════════════════════════════════════════════════════════╗
║  ✅ Done! NOW DO THIS:                                   ║
║                                                          ║
║  1. Click: Runtime → Restart Runtime                     ║
║  2. After restart, run cells in this order:              ║
║     Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5          ║
║                                                          ║
║  ⚠️  Do NOT skip the restart — old torch stays in        ║
║     memory until you restart.                            ║
╚══════════════════════════════════════════════════════════╝
""")

🔍 Checking current versions...
  Current PyTorch : 2.10.0+cu128
  Current CUDA    : 12.8

📥 Installing compatible versions...
  ✅ Uninstalled old moshi
  📥 Installing PyTorch 2.4.0 + CUDA 12.1 (takes ~2 min)...
  ✅ PyTorch 2.4.0 installed
  📥 Installing moshi from GitHub...
  ✅ moshi (GitHub latest) installed
  ✅ safetensors installed

╔══════════════════════════════════════════════════════════╗
║  ✅ Done! NOW DO THIS:                                   ║
║                                                          ║
║  1. Click: Runtime → Restart Runtime                     ║
║  2. After restart, run cells in this order:              ║
║     Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5          ║
║                                                          ║
║  ⚠️  Do NOT skip the restart — old torch stays in        ║
║     memory until you restart.                            ║
╚══════════════════════════════════════════════════════════╝



 Restart **Runtime**

In [1]:
import os
from datetime import datetime
from google.colab import drive

# --- Step 1: Mount Google Drive ---
print("📂 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("✅ Google Drive mounted successfully!\n")

# ============================================================
# FOLDER STRUCTURE DEFINITION
# All project files live under: MyDrive/Moshiko_Project/
# ============================================================
BASE_DIR = '/content/drive/MyDrive/Moshiko_Project'

FOLDERS = {
    'base'        : BASE_DIR,
    'models'      : f'{BASE_DIR}/models',           # Downloaded model weights
    'models_bf16' : f'{BASE_DIR}/models/bf16',      # Full precision weights
    'models_q8'   : f'{BASE_DIR}/models/q8',        # INT8 quantized weights
    'checkpoints' : f'{BASE_DIR}/checkpoints',      # Session resume points
    'outputs'     : f'{BASE_DIR}/outputs',           # All test results
    'audio_in'    : f'{BASE_DIR}/outputs/audio_input',   # Input audio files
    'audio_out'   : f'{BASE_DIR}/outputs/audio_output',  # Generated audio
    'benchmarks'  : f'{BASE_DIR}/outputs/benchmarks',    # Latency/speed results
    'comparisons' : f'{BASE_DIR}/outputs/comparisons',   # bf16 vs q8 results
    'logs'        : f'{BASE_DIR}/logs',             # Session logs
    'notebooks'   : f'{BASE_DIR}/notebooks',        # Notebook backups
}

# --- Step 2: Create all folders ---
print("📁 Creating project folder structure...")
for name, path in FOLDERS.items():
    os.makedirs(path, exist_ok=True)
    print(f"   {'✅ Exists' if os.path.exists(path) else '🆕 Created'}: {path.replace(BASE_DIR, 'Moshiko_Project')}")

# --- Step 3: Write a session log ---
session_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
log_file = f"{FOLDERS['logs']}/session_log.txt"
with open(log_file, 'a') as f:
    f.write(f"\n[{session_time}] New session started in Google Colab")

# --- Step 4: Update notebook timestamp in README ---
readme_path = f"{BASE_DIR}/README.txt"
readme_content = f"""
╔══════════════════════════════════════════════════════════╗
║           MOSHIKO RESEARCH PROJECT — README              ║
╠══════════════════════════════════════════════════════════╣
║  Last Session : {session_time}               ║
╠══════════════════════════════════════════════════════════╣
║  FOLDER STRUCTURE                                        ║
║                                                          ║
║  Moshiko_Project/                                        ║
║  ├── models/                                             ║
║  │   ├── bf16/     ← Full precision model weights        ║
║  │   └── q8/       ← INT8 quantized model weights        ║
║  ├── checkpoints/  ← Session resume points               ║
║  ├── outputs/                                            ║
║  │   ├── audio_input/   ← Your test audio files          ║
║  │   ├── audio_output/  ← Generated speech outputs       ║
║  │   ├── benchmarks/    ← Latency & speed results        ║
║  │   └── comparisons/   ← bf16 vs q8 quality tests       ║
║  ├── logs/         ← Session activity logs               ║
║  └── notebooks/    ← Saved notebook backups              ║
╠══════════════════════════════════════════════════════════╣
║  HOW TO RESUME AFTER DISCONNECT:                         ║
║  1. Re-run Cell 1 (Drive mount)                          ║
║  2. Re-run Cell 2 (GPU check)                            ║
║  3. Re-run Cell 3 (Install packages)                     ║
║  4. Run Cell 4 (will auto-skip downloaded files)         ║
║  5. Re-run Cell 5 (Load model)                           ║
║  6. Run any test cell you need                           ║
╚══════════════════════════════════════════════════════════╝
"""
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"""
╔══════════════════════════════════════════════╗
║  ✅ CELL 1 COMPLETE                          ║
║  📁 Project root: MyDrive/Moshiko_Project/   ║
║  📝 README.txt created/updated               ║
║  📋 Session logged at {session_time[:10]}           ║
╚══════════════════════════════════════════════╝
""")

# Make FOLDERS accessible to all other cells
import builtins
builtins.FOLDERS = FOLDERS
builtins.BASE_DIR = BASE_DIR
print("➡️  Ready for Cell 2")

📂 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully!

📁 Creating project folder structure...
   ✅ Exists: Moshiko_Project
   ✅ Exists: Moshiko_Project/models
   ✅ Exists: Moshiko_Project/models/bf16
   ✅ Exists: Moshiko_Project/models/q8
   ✅ Exists: Moshiko_Project/checkpoints
   ✅ Exists: Moshiko_Project/outputs
   ✅ Exists: Moshiko_Project/outputs/audio_input
   ✅ Exists: Moshiko_Project/outputs/audio_output
   ✅ Exists: Moshiko_Project/outputs/benchmarks
   ✅ Exists: Moshiko_Project/outputs/comparisons
   ✅ Exists: Moshiko_Project/logs
   ✅ Exists: Moshiko_Project/notebooks

╔══════════════════════════════════════════════╗
║  ✅ CELL 1 COMPLETE                          ║
║  📁 Project root: MyDrive/Moshiko_Project/   ║
║  📝 README.txt created/updated               ║
║  📋 Session logged at 2026-04-04           ║
╚══════════════════════════════════

In [2]:
import subprocess
import sys
import json
import os

print("🔍 Checking environment...\n")

# --- GPU Check ---
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        vram_free  = (torch.cuda.get_device_properties(0).total_memory
                      - torch.cuda.memory_allocated(0)) / 1e9
        print(f"  GPU      : {gpu_name}")
        print(f"  VRAM     : {vram_total:.1f} GB total | {vram_free:.1f} GB free")
    else:
        print("  ❌ NO GPU DETECTED!")
        print("  ⚠️  Go to: Runtime → Change runtime type → T4 GPU")
        raise SystemExit("Please enable GPU and re-run.")
except ImportError:
    gpu_name  = "Unknown (torch not installed yet)"
    vram_total = 0
    vram_free  = 0
    print("  ⚠️  PyTorch not installed yet — will check after Cell 3")

# --- Python Version ---
py_version = sys.version.split()[0]
print(f"  Python   : {py_version}")
py_ok = tuple(int(x) for x in py_version.split('.')[:2]) >= (3, 10)
print(f"  Python OK: {'✅ Yes' if py_ok else '❌ Needs 3.10+'}")

# --- Decide which model to use based on VRAM ---
if vram_total > 0:
    if vram_total >= 35:
        MODEL_PRECISION = 'bf16'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-bf16'
        precision_note  = 'Full precision (A100 detected)'
    elif vram_total >= 14:
        MODEL_PRECISION = 'q8'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
        precision_note  = 'INT8 quantized (T4 safe — uses ~14GB)'
    else:
        MODEL_PRECISION = 'q8'
        MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
        precision_note  = '⚠️ Low VRAM — using INT8, may be tight'
else:
    MODEL_PRECISION = 'q8'
    MODEL_REPO      = 'kyutai/moshiko-pytorch-q8'
    precision_note  = 'Default (torch not yet installed)'

# Save config for later cells
builtins.MODEL_PRECISION = MODEL_PRECISION
builtins.MODEL_REPO      = MODEL_REPO

# --- Save environment info to Drive ---
try:
    env_info = {
        'session_time'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'gpu_name'         : gpu_name,
        'vram_total_gb'    : round(vram_total, 2),
        'vram_free_gb'     : round(vram_free,  2),
        'python_version'   : py_version,
        'selected_model'   : MODEL_REPO,
        'model_precision'  : MODEL_PRECISION,
    }
    env_path = f"{FOLDERS['logs']}/environment_info.json"
    with open(env_path, 'w') as f:
        json.dump(env_info, f, indent=2)
    print(f"\n  💾 Environment info saved to Drive")
except Exception as e:
    print(f"  ⚠️  Could not save env info: {e} (run Cell 1 first)")

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 2 COMPLETE                                  ║
║  🎯 Selected model : {MODEL_REPO:<32}║
║  📊 Precision      : {MODEL_PRECISION:<32}║
║  📝 Note           : {precision_note:<32}║
╚══════════════════════════════════════════════════════╝
""")
print("➡️  Ready for Cell 3")

🔍 Checking environment...

  GPU      : Tesla T4
  VRAM     : 15.6 GB total | 15.6 GB free
  Python   : 3.12.13
  Python OK: ✅ Yes

  💾 Environment info saved to Drive

╔══════════════════════════════════════════════════════╗
║  ✅ CELL 2 COMPLETE                                  ║
║  🎯 Selected model : kyutai/moshiko-pytorch-q8       ║
║  📊 Precision      : q8                              ║
║  📝 Note           : INT8 quantized (T4 safe — uses ~14GB)║
╚══════════════════════════════════════════════════════╝

➡️  Ready for Cell 3


In [3]:
# ============================================================
# CELL 3 — Install Dependencies
# ============================================================
# PURPOSE: Install moshi, huggingface_hub, torchaudio, and
#          all required packages.
#
# ⏱️  Takes ~2-3 minutes on first run
# ✅ Safe to re-run (pip skips already-installed packages)
# ============================================================

import subprocess, sys, importlib

def install(package, import_name=None):
    """Install a package and verify it imported correctly."""
    name = import_name or package.split('==')[0].replace('-','_')
    try:
        importlib.import_module(name)
        print(f"  ✅ Already installed : {package}")
    except ImportError:
        print(f"  📥 Installing        : {package} ...", end='', flush=True)
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', package],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(" Done ✅")
        else:
            print(f" FAILED ❌\n    Error: {result.stderr[:200]}")

print("📦 Installing required packages...\n")

packages = [
    ('moshi',          'moshi'),
    ('huggingface_hub','huggingface_hub'),
    ('torchaudio',     'torchaudio'),
    ('soundfile',      'soundfile'),
    ('numpy',          'numpy'),
    ('scipy',          'scipy'),
    ('matplotlib',     'matplotlib'),
    ('IPython',        'IPython'),
    ('tqdm',           'tqdm'),
]

for pkg, imp in packages:
    install(pkg, imp)

# --- Verify critical imports ---
print("\n🔍 Verifying critical imports...")
critical = ['torch', 'moshi', 'huggingface_hub', 'torchaudio', 'soundfile']
all_ok = True
for mod in critical:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'unknown')
        print(f"  ✅ {mod:<20} v{ver}")
    except ImportError:
        print(f"  ❌ {mod:<20} IMPORT FAILED — re-run this cell")
        all_ok = False

if all_ok:
    print("""
╔══════════════════════════════════════════╗
║  ✅ CELL 3 COMPLETE — All packages OK   ║
╚══════════════════════════════════════════╝
""")
    print("➡️  Ready for Cell 4")
else:
    print("""
╔══════════════════════════════════════════╗
║  ❌ Some packages failed — re-run cell  ║
╚══════════════════════════════════════════╝
""")

📦 Installing required packages...

  ✅ Already installed : moshi
  ✅ Already installed : huggingface_hub
  ✅ Already installed : torchaudio
  ✅ Already installed : soundfile
  ✅ Already installed : numpy
  ✅ Already installed : scipy
  ✅ Already installed : matplotlib
  ✅ Already installed : IPython
  ✅ Already installed : tqdm

🔍 Verifying critical imports...
  ✅ torch                v2.4.0+cu121
  ✅ moshi                v0.2.13
  ✅ huggingface_hub      v0.36.2
  ✅ torchaudio           v2.4.0+cu121
  ✅ soundfile            v0.13.1

╔══════════════════════════════════════════╗
║  ✅ CELL 3 COMPLETE — All packages OK   ║
╚══════════════════════════════════════════╝

➡️  Ready for Cell 4


In [4]:
import os, json, time
from datetime import datetime
from huggingface_hub import hf_hub_download, login
from tqdm import tqdm

# ---- Optional: Authenticate if repos are gated ----
# login()  # Uncomment if you get 401 errors or repos are private

# ---- Checkpoint helpers ----
CHECKPOINT_FILE = f"{FOLDERS['checkpoints']}/download_checkpoint.json"

def load_checkpoint():
    """Load which files have already been downloaded."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {'downloaded': {}, 'last_updated': None}

def save_checkpoint(ckpt):
    """Save progress after each file download."""
    ckpt['last_updated'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    os.makedirs(os.path.dirname(CHECKPOINT_FILE), exist_ok=True)
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(ckpt, f, indent=2)

def download_file(repo_id, filename, local_dir, ckpt, label):
    """
    Download a single file from HuggingFace Hub.
    Skips if already downloaded (checked via checkpoint).
    """
    key = f"{repo_id}/{filename}"
    local_path = os.path.join(local_dir, filename)

    # Check checkpoint
    if key in ckpt['downloaded'] and os.path.exists(local_path):
        size_mb = os.path.getsize(local_path) / 1e6
        print(f"  ⏭️  SKIP (already downloaded): {label} ({size_mb:.0f} MB)")
        return local_path

    print(f"  📥 Downloading: {label} ...", flush=True)
    start = time.time()
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=local_dir,
            local_dir_use_symlinks=False,
        )
        elapsed  = time.time() - start
        size_mb  = os.path.getsize(path) / 1e6
        print(f"     ✅ Done — {size_mb:.0f} MB in {elapsed:.0f}s")

        # Save to checkpoint
        ckpt['downloaded'][key] = {
            'path': path,
            'size_mb': round(size_mb, 1),
            'downloaded_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        save_checkpoint(ckpt)
        return path
    except Exception as e:
        print(f"     ❌ FAILED: {e}")
        print(f"     💡 Re-run this cell to retry.")
        return None

# ---- Load existing checkpoint ----
ckpt = load_checkpoint()
if ckpt['last_updated']:
    print(f"📋 Checkpoint found — last updated: {ckpt['last_updated']}")
    print(f"   Already downloaded: {len(ckpt['downloaded'])} file(s)\n")
else:
    print("📋 No checkpoint found — starting fresh download\n")

# ---- Files to download ----
# ✅ CORRECTED FILENAMES based on actual HuggingFace repo contents
# Repo: https://huggingface.co/kyutai/moshiko-pytorch-q8
# Repo: https://huggingface.co/kyutai/moshiko-pytorch-bf16

files_to_download = [
    # ── Q8 models (primary — optimized for T4 GPU, ~8.4 GB total) ──
    ('kyutai/moshiko-pytorch-q8',
     'model.q8.safetensors',
     FOLDERS['models_q8'],
     'Mimi Codec (q8)'),

    ('kyutai/moshiko-pytorch-q8',
     'tokenizer-e351c8d8-checkpoint125.safetensors',
     FOLDERS['models_q8'],
     'Moshiko LM (q8)'),

    # ── BF16 Mimi only (for comparison — ~385 MB) ──
    ('kyutai/moshiko-pytorch-bf16',
     'tokenizer-e351c8d8-checkpoint125.safetensors',
     FOLDERS['models_bf16'],
     'Mimi Codec (bf16)'),

    # ── Optional: BF16 Moshiko LM (uncomment if needed — ~15.4 GB) ──
    # ('kyutai/moshiko-pytorch-bf16',
    #  'model.safetensors',
    #  FOLDERS['models_bf16'],
    #  'Moshiko LM (bf16)'),
]

print("⬇️  Starting downloads...\n")
downloaded_paths = {}
for repo, fname, local_dir, label in files_to_download:
    # Ensure target directory exists
    os.makedirs(local_dir, exist_ok=True)
    path = download_file(repo, fname, local_dir, ckpt, label)
    if path:
        downloaded_paths[label] = path

# ---- Summary ----
total_files = len(ckpt['downloaded'])
total_size_mb = sum(info.get('size_mb', 0) for info in ckpt['downloaded'].values())

print(f"""
╔════════════════════════════════════════════════════════════════════════════════╗
║  ✅ CELL 4 COMPLETE                                                           ║
║  📦 Total files downloaded : {total_files} file(s){" " * 20}                  ║
║  💾 Total size on Drive    : {total_size_mb:>.0f} MB{" " * 22}                ║
║  💾 Checkpoint saved       : {CHECKPOINT_FILE.split('/')[-1]:<26}             ║
║                                                                               ║
║  📁 Download locations:                                                       ║
""")
for label, path in downloaded_paths.items():
    size = os.path.getsize(path) / 1e6
    print(f"     • {label:<25} → {size:>.0f} MB")
print(f"""
║                                                          ║
║  If this cell was interrupted:                           ║
║  → Re-run Cell 4 — it will resume where it stopped       ║
╚══════════════════════════════════════════════════════════╝
""")
print("➡️  Ready for Cell 5")

📋 Checkpoint found — last updated: 2026-04-04 07:19:24
   Already downloaded: 3 file(s)

⬇️  Starting downloads...

  ⏭️  SKIP (already downloaded): Mimi Codec (q8) (8009 MB)
  ⏭️  SKIP (already downloaded): Moshiko LM (q8) (385 MB)
  ⏭️  SKIP (already downloaded): Mimi Codec (bf16) (385 MB)

╔════════════════════════════════════════════════════════════════════════════════╗
║  ✅ CELL 4 COMPLETE                                                           ║
║  📦 Total files downloaded : 3 file(s)                                      ║
║  💾 Total size on Drive    : 8778 MB                                      ║
║  💾 Checkpoint saved       : download_checkpoint.json               ║
║                                                                               ║
║  📁 Download locations:                                                       ║

     • Mimi Codec (q8)           → 8009 MB
     • Moshiko LM (q8)           → 385 MB
     • Mimi Codec (bf16)         → 385 MB

║                      

In [5]:
import torch
print(f"PyTorch version: {torch.__version__}")
assert torch.__version__.startswith('2.4'), \
    f"❌ Wrong PyTorch: {torch.__version__} — re-run Cell 3b then restart runtime"
print("✅ PyTorch 2.4 confirmed — safe to load model")


import torch, time, os, builtins
from moshi.models import loaders, LMGen

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")
if DEVICE == 'cpu':
    print("  ⚠️  WARNING: CPU mode will be very slow!")

# ---- Locate model files ----
Q8_DIR   = FOLDERS['models_q8']
BF16_DIR = FOLDERS['models_bf16']

mimi_path  = os.path.join(Q8_DIR, 'tokenizer-e351c8d8-checkpoint125.safetensors')
moshi_path = os.path.join(Q8_DIR, 'model.q8.safetensors')

# Check files exist
for label, path in [('Mimi Codec', mimi_path), ('Moshiko LM (q8)', moshi_path)]:
    if not os.path.exists(path):
        print(f"  ❌ Missing: {label} at {path}")
        print("  → Re-run Cell 4 to download missing files.")
        raise FileNotFoundError(f"Missing: {path}")
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Load Mimi (Audio Codec) ----
print("\n⏳ Loading Mimi audio codec...")
start = time.time()
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.set_num_codebooks(8)
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- Load Moshiko LM ----
# CORRECT function: get_moshi_lm()  (NOT get_moshi or load_moshi)
print("\n⏳ Loading Moshiko language model (q8)...")
start = time.time()
moshi_lm = loaders.get_moshi_lm(moshi_path, device=DEVICE)
moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- Wrap in LMGen (required for streaming inference) ----
# LMGen handles sampling parameters for generation
print("\n⏳ Wrapping in LMGen (streaming generator)...")
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
print("  ✅ LMGen ready")

# ---- VRAM usage ----
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"\n  📊 VRAM used    : {allocated:.1f} GB")
    print(f"  📊 VRAM reserved: {reserved:.1f} GB")

# ---- Make globally accessible to all other cells ----
builtins.mimi     = mimi
builtins.moshi_lm = moshi_lm
builtins.lm_gen   = lm_gen
builtins.DEVICE   = DEVICE

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 5 COMPLETE — Models loaded!                 ║
║  🎵 Mimi codec   : Ready (audio tokenizer)           ║
║  🧠 Moshiko LM   : Ready (speech-text model)         ║
║  🎛️  LMGen        : Ready (streaming inference)       ║
║  🖥️  Device       : {DEVICE:<34}║
╚══════════════════════════════════════════════════════╝
""")
print("➡️  Ready for TEST Cells (6, 7, 8, 9)")

PyTorch version: 2.4.0+cu121
✅ PyTorch 2.4 confirmed — safe to load model
🖥️  Using device: cuda
  ✅ Found: Mimi Codec (385 MB)
  ✅ Found: Moshiko LM (q8) (8009 MB)

⏳ Loading Mimi audio codec...
  ✅ Mimi loaded in 1.1s

⏳ Loading Moshiko language model (q8)...


RuntimeError: Error(s) in loading state_dict for LMModel:
	Unexpected key(s) in state_dict: "text_linear.weight_scb", "transformer.layers.0.self_attn.out_projs.0.weight_scb", "transformer.layers.0.self_attn.in_projs.0.weight_scb", "transformer.layers.0.gating.linear_in.weight_scb", "transformer.layers.0.gating.linear_out.weight_scb", "transformer.layers.1.self_attn.out_projs.0.weight_scb", "transformer.layers.1.self_attn.in_projs.0.weight_scb", "transformer.layers.1.gating.linear_in.weight_scb", "transformer.layers.1.gating.linear_out.weight_scb", "transformer.layers.2.self_attn.out_projs.0.weight_scb", "transformer.layers.2.self_attn.in_projs.0.weight_scb", "transformer.layers.2.gating.linear_in.weight_scb", "transformer.layers.2.gating.linear_out.weight_scb", "transformer.layers.3.self_attn.out_projs.0.weight_scb", "transformer.layers.3.self_attn.in_projs.0.weight_scb", "transformer.layers.3.gating.linear_in.weight_scb", "transformer.layers.3.gating.linear_out.weight_scb", "transform

In [6]:
# CELL 3c — Install bitsandbytes (required for q8 model)
import subprocess, sys

print("📥 Installing bitsandbytes for INT8 quantized model support...")

result = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'bitsandbytes>=0.41.0'
], capture_output=True, text=True)

if result.returncode == 0:
    print("  ✅ bitsandbytes installed")
else:
    print(f"  ❌ Failed:\n{result.stderr[-300:]}")

# Verify it works
try:
    import bitsandbytes as bnb
    print(f"  ✅ bitsandbytes version: {bnb.__version__}")
except ImportError:
    print("  ❌ Import failed — try running this cell again")

print("""
╔══════════════════════════════════════════════════════════╗
║  NOW DO:                                                 ║
║  Runtime → Restart Runtime                               ║
║  Then run: Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5   ║
╚══════════════════════════════════════════════════════════╝
""")

📥 Installing bitsandbytes for INT8 quantized model support...
  ✅ bitsandbytes installed
  ✅ bitsandbytes version: 0.49.2

╔══════════════════════════════════════════════════════════╗
║  NOW DO:                                                 ║
║  Runtime → Restart Runtime                               ║
║  Then run: Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5   ║
╚══════════════════════════════════════════════════════════╝



In [7]:
import torch
print(f"PyTorch version: {torch.__version__}")
assert torch.__version__.startswith('2.4'), \
    f"❌ Wrong PyTorch: {torch.__version__} — re-run Cell 3b then restart runtime"
print("✅ PyTorch 2.4 confirmed — safe to load model")


import torch, time, os, builtins
from moshi.models import loaders, LMGen

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")
if DEVICE == 'cpu':
    print("  ⚠️  WARNING: CPU mode will be very slow!")

# ---- Locate model files ----
Q8_DIR   = FOLDERS['models_q8']
BF16_DIR = FOLDERS['models_bf16']

mimi_path  = os.path.join(Q8_DIR, 'tokenizer-e351c8d8-checkpoint125.safetensors')
moshi_path = os.path.join(Q8_DIR, 'model.q8.safetensors')

# Check files exist
for label, path in [('Mimi Codec', mimi_path), ('Moshiko LM (q8)', moshi_path)]:
    if not os.path.exists(path):
        print(f"  ❌ Missing: {label} at {path}")
        print("  → Re-run Cell 4 to download missing files.")
        raise FileNotFoundError(f"Missing: {path}")
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Load Mimi (Audio Codec) ----
print("\n⏳ Loading Mimi audio codec...")
start = time.time()
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.set_num_codebooks(8)
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- Load Moshiko LM ----
# CORRECT function: get_moshi_lm()  (NOT get_moshi or load_moshi)
# ---- Load Moshiko LM (q8 — bitsandbytes-aware) ----
print("\n⏳ Loading Moshiko language model (q8)...")
start = time.time()

# Tell the loader to use quantize=True so it handles _scb keys correctly
moshi_lm = loaders.get_moshi_lm(
    moshi_path,
    device=DEVICE,
    lm_kwargs={'quantize': True}   # ← this tells loader to expect INT8 weights
)
moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- Wrap in LMGen (required for streaming inference) ----
# LMGen handles sampling parameters for generation
print("\n⏳ Wrapping in LMGen (streaming generator)...")
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
print("  ✅ LMGen ready")

# ---- VRAM usage ----
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f"\n  📊 VRAM used    : {allocated:.1f} GB")
    print(f"  📊 VRAM reserved: {reserved:.1f} GB")

# ---- Make globally accessible to all other cells ----
builtins.mimi     = mimi
builtins.moshi_lm = moshi_lm
builtins.lm_gen   = lm_gen
builtins.DEVICE   = DEVICE

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 5 COMPLETE — Models loaded!                 ║
║  🎵 Mimi codec   : Ready (audio tokenizer)           ║
║  🧠 Moshiko LM   : Ready (speech-text model)         ║
║  🎛️  LMGen        : Ready (streaming inference)       ║
║  🖥️  Device       : {DEVICE:<34}║
╚══════════════════════════════════════════════════════╝
""")
print("➡️  Ready for TEST Cells (6, 7, 8, 9)")

PyTorch version: 2.4.0+cu121
✅ PyTorch 2.4 confirmed — safe to load model
🖥️  Using device: cuda
  ✅ Found: Mimi Codec (385 MB)
  ✅ Found: Moshiko LM (q8) (8009 MB)

⏳ Loading Mimi audio codec...
  ✅ Mimi loaded in 0.9s

⏳ Loading Moshiko language model (q8)...


AssertionError: expected 9 delays, got 1.

In [8]:
# ============================================================
# CELL 5 — FIXED: Load via config.json (correct approach)
# ============================================================
import torch, time, os, builtins, json
from moshi.models import loaders, LMGen

# Version check
assert torch.__version__.startswith('2.4'), f"Wrong PyTorch: {torch.__version__}"
print(f"✅ PyTorch {torch.__version__} confirmed\n")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")

# ---- Locate files ----
Q8_DIR     = FOLDERS['models_q8']
mimi_path  = os.path.join(Q8_DIR, 'tokenizer-e351c8d8-checkpoint125.safetensors')
moshi_path = os.path.join(Q8_DIR, 'model.q8.safetensors')
config_path = os.path.join(Q8_DIR, 'config.json')

for label, path in [('Mimi Codec', mimi_path), ('Moshiko LM (q8)', moshi_path)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {label} at {path}\n→ Re-run Cell 4")
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Check if config.json exists (download if not) ----
if not os.path.exists(config_path):
    print("\n  📥 config.json not found — downloading...")
    from huggingface_hub import hf_hub_download
    config_path = hf_hub_download(
        repo_id='kyutai/moshiko-pytorch-q8',
        filename='config.json',
        local_dir=Q8_DIR,
        local_dir_use_symlinks=False,
    )
    print(f"  ✅ config.json downloaded")
else:
    print(f"  ✅ Found: config.json")

# ---- Read config to see what parameters the model needs ----
with open(config_path, 'r') as f:
    config = json.load(f)
print(f"\n  📋 Model config loaded")
print(f"     delays  : {config.get('delays', 'not found')}")
print(f"     n_q     : {config.get('n_q', 'not found')}")
print(f"     dep_q   : {config.get('dep_q', 'not found')}")
print(f"     quantize: {config.get('quantize', 'not found')}")

# ---- Load Mimi ----
print("\n⏳ Loading Mimi audio codec...")
start = time.time()
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.set_num_codebooks(8)
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- Load Moshiko LM using config.json parameters ----
print("\n⏳ Loading Moshiko LM using config.json...")
start = time.time()

# Pass the config as lm_kwargs so loader uses correct architecture
moshi_lm = loaders.get_moshi_lm(
    moshi_path,
    device=DEVICE,
    lm_kwargs=config,           # ← pass full config so delays/n_q are correct
)
moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- Wrap in LMGen ----
print("\n⏳ Wrapping in LMGen...")
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
print("  ✅ LMGen ready")

# ---- VRAM ----
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    print(f"\n  📊 VRAM used    : {torch.cuda.memory_allocated()/1e9:.1f} GB")
    print(f"  📊 VRAM reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB")

builtins.mimi     = mimi
builtins.moshi_lm = moshi_lm
builtins.lm_gen   = lm_gen
builtins.DEVICE   = DEVICE

print("""
╔══════════════════════════════════════════════════╗
║  ✅ CELL 5 COMPLETE — Models loaded!             ║
╚══════════════════════════════════════════════════╝
""")

✅ PyTorch 2.4.0+cu121 confirmed

🖥️  Using device: cuda
  ✅ Found: Mimi Codec (385 MB)
  ✅ Found: Moshiko LM (q8) (8009 MB)

  📥 config.json not found — downloading...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

  ✅ config.json downloaded

  📋 Model config loaded
     delays  : [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1]
     n_q     : 16
     dep_q   : 8
     quantize: True

⏳ Loading Mimi audio codec...
  ✅ Mimi loaded in 0.8s

⏳ Loading Moshiko LM using config.json...


TypeError: StreamingTransformerLayer.__init__() got an unexpected keyword argument 'moshi_name'

In [9]:
# ============================================================
# CELL 5 — FIXED: Pass only required config keys
# ============================================================
import torch, time, os, builtins, json
from moshi.models import loaders, LMGen

assert torch.__version__.startswith('2.4'), f"Wrong PyTorch: {torch.__version__}"
print(f"✅ PyTorch {torch.__version__} confirmed\n")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")

# ---- Locate files ----
Q8_DIR      = FOLDERS['models_q8']
mimi_path   = os.path.join(Q8_DIR, 'tokenizer-e351c8d8-checkpoint125.safetensors')
moshi_path  = os.path.join(Q8_DIR, 'model.q8.safetensors')

for label, path in [('Mimi Codec', mimi_path), ('Moshiko LM (q8)', moshi_path)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {label}\n→ Re-run Cell 4")
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Load Mimi ----
print("\n⏳ Loading Mimi audio codec...")
start = time.time()
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.set_num_codebooks(8)
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- Load Moshiko LM ----
# We now know from config.json exactly what the model needs.
# We pass ONLY the keys LMModel.__init__ accepts.
# Extra keys like 'moshi_name', 'model_type' etc. cause TypeError — excluded.
print("\n⏳ Loading Moshiko LM (q8)...")
start = time.time()

lm_kwargs = {
    'delays'  : [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    'n_q'     : 16,
    'dep_q'   : 8,
    'quantize': True,
}

moshi_lm = loaders.get_moshi_lm(
    moshi_path,
    device=DEVICE,
    lm_kwargs=lm_kwargs,
)
moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- Wrap in LMGen ----
print("\n⏳ Creating LMGen wrapper...")
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
print("  ✅ LMGen ready")

# ---- VRAM usage ----
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    print(f"\n  📊 VRAM used    : {torch.cuda.memory_allocated()/1e9:.1f} GB")
    print(f"  📊 VRAM reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB")

# ---- Store globally ----
builtins.mimi     = mimi
builtins.moshi_lm = moshi_lm
builtins.lm_gen   = lm_gen
builtins.DEVICE   = DEVICE

print("""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 5 COMPLETE — All models loaded!             ║
║  🎵 Mimi   : Ready                                   ║
║  🧠 Moshiko: Ready                                   ║
║  🎛️  LMGen  : Ready                                   ║
╚══════════════════════════════════════════════════════╝
""")

✅ PyTorch 2.4.0+cu121 confirmed

🖥️  Using device: cuda
  ✅ Found: Mimi Codec (385 MB)
  ✅ Found: Moshiko LM (q8) (8009 MB)

⏳ Loading Mimi audio codec...
  ✅ Mimi loaded in 0.8s

⏳ Loading Moshiko LM (q8)...


TypeError: StreamingTransformer.__init__() missing 1 required positional argument: 'num_layers'

In [10]:
# ============================================================
# CELL 5a — Read full config to find ALL required parameters
# ============================================================
# Run this FIRST to see all config values, then run Cell 5b
# ============================================================
import json, os

config_path = os.path.join(FOLDERS['models_q8'], 'config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

print("📋 Full config.json contents:")
print("=" * 50)
for k, v in config.items():
    print(f"  {k:<40} : {v}")
print("=" * 50)

📋 Full config.json contents:
  moshi_name                               : model.q8.safetensors
  mimi_name                                : tokenizer-e351c8d8-checkpoint125.safetensors
  tokenizer_name                           : tokenizer_spm_32k_3.model
  quantize                                 : True
  dim                                      : 4096
  text_card                                : 32000
  existing_text_padding_id                 : 3
  n_q                                      : 16
  dep_q                                    : 8
  card                                     : 2048
  num_heads                                : 32
  num_layers                               : 32
  hidden_scale                             : 4.125
  causal                                   : True
  layer_scale                              : None
  context                                  : 3000
  max_period                               : 10000
  gating                                   : silu
  n

In [11]:
# ============================================================
# CELL 5 — FINAL FIXED VERSION (all parameters from config)
# ============================================================
import torch, time, os, builtins
from moshi.models import loaders, LMGen

assert torch.__version__.startswith('2.4'), f"Wrong PyTorch: {torch.__version__}"
print(f"✅ PyTorch {torch.__version__} confirmed\n")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {DEVICE}")

# ---- Locate files ----
Q8_DIR     = FOLDERS['models_q8']
mimi_path  = os.path.join(Q8_DIR, 'tokenizer-e351c8d8-checkpoint125.safetensors')
moshi_path = os.path.join(Q8_DIR, 'model.q8.safetensors')

for label, path in [('Mimi Codec', mimi_path), ('Moshiko LM (q8)', moshi_path)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing: {label}\n→ Re-run Cell 4")
    print(f"  ✅ Found: {label} ({os.path.getsize(path)/1e6:.0f} MB)")

# ---- Load Mimi ----
print("\n⏳ Loading Mimi audio codec...")
start = time.time()
mimi = loaders.get_mimi(mimi_path, device=DEVICE)
mimi.set_num_codebooks(8)
mimi.eval()
print(f"  ✅ Mimi loaded in {time.time()-start:.1f}s")

# ---- All parameters from config.json ----
lm_kwargs = {
    # Core architecture
    'delays'                        : [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    'n_q'                           : 16,
    'dep_q'                         : 8,
    'card'                          : 2048,
    'text_card'                     : 32000,
    'existing_text_padding_id'      : 3,
    'dim'                           : 4096,
    'num_heads'                     : 32,
    'num_layers'                    : 32,
    'hidden_scale'                  : 4.125,
    'causal'                        : True,
    'context'                       : 3000,
    'max_period'                    : 10000,
    'gating'                        : 'silu',
    'norm'                          : 'rms_norm_f32',
    'positional_embedding'          : 'rope',
    'layer_scale'                   : None,
    # Depth transformer
    'depformer_dim'                 : 1024,
    'depformer_dim_feedforward'     : 4224,
    'depformer_num_heads'           : 16,
    'depformer_num_layers'          : 6,
    'depformer_causal'              : True,
    'depformer_layer_scale'         : None,
    'depformer_multi_linear'        : True,
    'depformer_context'             : 8,
    'depformer_max_period'          : 10000,
    'depformer_gating'              : 'silu',
    'depformer_pos_emb'             : 'none',
    'depformer_weights_per_step'    : True,
    # Quantization
    'quantize'                      : True,
}

# ---- Load Moshiko LM ----
print("\n⏳ Loading Moshiko LM (q8) — this takes ~2-3 min on T4...")
start = time.time()
moshi_lm = loaders.get_moshi_lm(
    moshi_path,
    device=DEVICE,
    lm_kwargs=lm_kwargs,
)
moshi_lm.eval()
print(f"  ✅ Moshiko LM loaded in {time.time()-start:.1f}s")

# ---- Wrap in LMGen ----
print("\n⏳ Creating LMGen wrapper...")
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
print("  ✅ LMGen ready")

# ---- VRAM usage ----
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    print(f"\n  📊 VRAM used    : {torch.cuda.memory_allocated()/1e9:.1f} GB")
    print(f"  📊 VRAM reserved: {torch.cuda.memory_reserved()/1e9:.1f} GB")

# ---- Store globally ----
builtins.mimi     = mimi
builtins.moshi_lm = moshi_lm
builtins.lm_gen   = lm_gen
builtins.DEVICE   = DEVICE

print("""
╔══════════════════════════════════════════════════════╗
║  ✅ CELL 5 COMPLETE — All models loaded!             ║
║  🎵 Mimi   : Ready (audio tokenizer)                 ║
║  🧠 Moshiko: Ready (7B speech-text LM)               ║
║  🎛️  LMGen  : Ready (streaming inference)             ║
╚══════════════════════════════════════════════════════╝
""")
print("➡️  Ready for TEST Cells (6, 7, 8, 9)")

✅ PyTorch 2.4.0+cu121 confirmed

🖥️  Using device: cuda
  ✅ Found: Mimi Codec (385 MB)
  ✅ Found: Moshiko LM (q8) (8009 MB)

⏳ Loading Mimi audio codec...
  ✅ Mimi loaded in 1.2s

⏳ Loading Moshiko LM (q8) — this takes ~2-3 min on T4...
  ✅ Moshiko LM loaded in 41.1s

⏳ Creating LMGen wrapper...
  ✅ LMGen ready

  📊 VRAM used    : 8.4 GB
  📊 VRAM reserved: 8.5 GB

╔══════════════════════════════════════════════════════╗
║  ✅ CELL 5 COMPLETE — All models loaded!             ║
║  🎵 Mimi   : Ready (audio tokenizer)                 ║
║  🧠 Moshiko: Ready (7B speech-text LM)               ║
║  🎛️  LMGen  : Ready (streaming inference)             ║
╚══════════════════════════════════════════════════════╝

➡️  Ready for TEST Cells (6, 7, 8, 9)


In [ ]:
import torch, torchaudio, soundfile as sf
import numpy as np, time, os
from IPython.display import Audio, display
from datetime import datetime

SAMPLE_RATE = 24000   # Mimi operates at 24kHz
TEST_DURATION = 3     # seconds of synthetic test audio

# ---- Step 1: Create a synthetic test audio clip ----
# Using a 440Hz tone (A note) as a simple, reproducible test signal.
# In a real experiment, replace this with actual speech audio.
print("🎵 Creating synthetic test audio (440Hz tone, 3 seconds)...")
t   = torch.linspace(0, TEST_DURATION, SAMPLE_RATE * TEST_DURATION)
wav = (0.3 * torch.sin(2 * np.pi * 440 * t)).unsqueeze(0).unsqueeze(0)  # [1, 1, T]
print(f"  Input shape: {wav.shape} | Sample rate: {SAMPLE_RATE} Hz")

# ---- Step 2: Save input audio to Drive ----
input_path = f"{FOLDERS['audio_in']}/test_input_440hz.wav"
sf.write(input_path, wav.squeeze().numpy(), SAMPLE_RATE)
print(f"  💾 Input saved: {input_path.replace(BASE_DIR, 'Moshiko_Project')}")

# ---- Step 3: Encode with Mimi (audio → tokens) ----
print("\n⚙️  Encoding audio → tokens (Mimi codec)...")
wav_gpu = wav.to(DEVICE)
encode_start = time.time()
with torch.no_grad():
    codes = mimi.encode(wav_gpu)    # [B, K=8, T_codes]
encode_time = time.time() - encode_start
print(f"  ✅ Encoded shape : {codes.shape}  (B × codebooks × time)")
print(f"  ⏱️  Encode time  : {encode_time*1000:.1f} ms")

# ---- Step 4: Decode tokens back to audio ----
print("\n🔊 Decoding tokens → audio...")
decode_start = time.time()
with torch.no_grad():
    decoded_wav = mimi.decode(codes)   # [B, 1, T]
decode_time = time.time() - decode_start
print(f"  ✅ Output shape : {decoded_wav.shape}")
print(f"  ⏱️  Decode time : {decode_time*1000:.1f} ms")

# ---- Step 5: Save output audio to Drive ----
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f"{FOLDERS['audio_out']}/test_output_{ts}.wav"
out_audio = decoded_wav.squeeze().cpu().numpy()
sf.write(output_path, out_audio, SAMPLE_RATE)
print(f"  💾 Output saved: {output_path.replace(BASE_DIR, 'Moshiko_Project')}")

# ---- Step 6: Play both in notebook ----
print("\n▶️  INPUT audio:")
display(Audio(wav.squeeze().numpy(), rate=SAMPLE_RATE))
print("▶️  OUTPUT audio (after Mimi encode→decode):")
display(Audio(out_audio, rate=SAMPLE_RATE))

# ---- Step 7: Quality metric — SNR (Signal-to-Noise Ratio) ----
# Trim to same length for comparison
min_len   = min(wav.squeeze().shape[0], decoded_wav.squeeze().cpu().shape[0])
original  = wav.squeeze().numpy()[:min_len]
reconstructed = out_audio[:min_len]
noise     = original - reconstructed
snr       = 10 * np.log10(np.mean(original**2) / (np.mean(noise**2) + 1e-8))
print(f"\n📊 Quality metric: SNR = {snr:.2f} dB")
print(f"   (Higher is better. >20dB = good reconstruction)")

# ---- Save result summary ----
result = {
    'test': 'Basic Audio Encode-Decode',
    'timestamp': ts,
    'input_shape': str(wav.shape),
    'output_shape': str(decoded_wav.shape),
    'encode_time_ms': round(encode_time * 1000, 2),
    'decode_time_ms': round(decode_time * 1000, 2),
    'snr_db': round(float(snr), 2),
    'input_file': input_path,
    'output_file': output_path,
}
import json
with open(f"{FOLDERS['outputs']}/test1_results_{ts}.json", 'w') as f:
    json.dump(result, f, indent=2)

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ TEST 1 COMPLETE                                  ║
║  🎵 Audio encoded & decoded successfully             ║
║  📊 SNR: {snr:.2f} dB                                   ║
║  ⏱️  Encode: {encode_time*1000:.0f}ms | Decode: {decode_time*1000:.0f}ms               ║
║  💾 Results saved to Drive                           ║
╚══════════════════════════════════════════════════════╝
""")